In [1]:
import os
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor
from itertools import product
import cupy as cp

C:\Users\Windows11\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\cupy\_environment.py:275: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


Phase 3: 

From what I did in the notebook before, add them first into this notebook.

Since we used nrows = 2000000, now we change our test size. 

In [2]:
# Read data
TRAIN_PATH = "train.csv" 
TEST_PATH  = "test.csv"
train = pd.read_csv(TRAIN_PATH, nrows= 10000000)
test  = pd.read_csv(TEST_PATH)

Add the features which already found or did in the previous notbook:

haversine_km : direct distance
manhattan_km : road-like Manhattan distance on lat/lon
trip_bearing: degrees

In [3]:
# add features:
def haversine_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371.0 * c

def manhattan_km(lon1, lat1, lon2, lat2):
    """
    Approximate road-like Manhattan distance on lat/lon:
    north-south leg + east-west leg
    """
    leg1 = haversine_km(lon1, lat1, lon1, lat2)  # move only in latitude
    leg2 = haversine_km(lon1, lat2, lon2, lat2)  # move only in longitude
    return leg1 + leg2

def trip_bearing(lon1, lat1, lon2, lat2):
    """
    Initial bearing (forward azimuth) from pickup to dropoff in degrees.
    Output range: [-180, 180]
    """
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    dlon = lon2 - lon1

    x = np.sin(dlon) * np.cos(lat2)
    y = (
        np.cos(lat1) * np.sin(lat2)
        - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    )

    bearing = np.degrees(np.arctan2(x, y))
    return bearing

# Define coordinates for points of interest (POIs) in NYC
POI_COORDS = {
    "jfk": (-73.7781, 40.6413),
    "lga": (-73.8740, 40.7769),
    "manhattan_center": (-73.9855, 40.7580),
}

def distance_to_poi(lon, lat, poi_lon, poi_lat):
    return haversine_km(lon, lat, poi_lon, poi_lat)

BASE_YEAR = 2009
BASE_MONTH = 2009 * 12 + 1

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # datetime
    out["pickup_datetime"] = pd.to_datetime(out["pickup_datetime"], errors="coerce", utc=True)
    out["pickup_hour"] = out["pickup_datetime"].dt.hour
    out["pickup_dow"] = out["pickup_datetime"].dt.dayofweek
    out["pickup_month"] = out["pickup_datetime"].dt.month

    # year-related features
    out["pickup_year"] = out["pickup_datetime"].dt.year
    out["year_index"] = out["pickup_year"] - BASE_YEAR
    out["year_month_index"] = (
        out["pickup_datetime"].dt.year * 12 + out["pickup_datetime"].dt.month
        - BASE_MONTH
    )

    out["is_weekend"] = (out["pickup_dow"] >= 5).astype(int)

    # rush hour flags
    out["is_rush_hour"] = out["pickup_hour"].isin([7,8,9,16,17,18,19]).astype(int)
    out["is_night"] = out["pickup_hour"].isin([22,23,0,1,2,3,4,5]).astype(int)

    # distance
    out["haversine_km"] = haversine_km(
        out["pickup_longitude"], out["pickup_latitude"],
        out["dropoff_longitude"], out["dropoff_latitude"]
    )

    out["manhattan_km"] = manhattan_km(
        out["pickup_longitude"], out["pickup_latitude"],
        out["dropoff_longitude"], out["dropoff_latitude"]
    )

    out["distance_ratio_manhattan_haversine"] = (
        out["manhattan_km"] / out["haversine_km"].replace(0, np.nan)
    )

    out["trip_bearing"] = trip_bearing(
        out["pickup_longitude"], out["pickup_latitude"],
        out["dropoff_longitude"], out["dropoff_latitude"]
    )

    bearing_rad = np.radians(out["trip_bearing"])
    out["bearing_sin"] = np.sin(bearing_rad)
    out["bearing_cos"] = np.cos(bearing_rad)

    # POI distance features
    jfk_lon, jfk_lat = POI_COORDS["jfk"]
    lga_lon, lga_lat = POI_COORDS["lga"]
    mh_lon, mh_lat = POI_COORDS["manhattan_center"]

    out["pickup_to_jfk_km"] = distance_to_poi(
        out["pickup_longitude"], out["pickup_latitude"], jfk_lon, jfk_lat
    )
    out["dropoff_to_jfk_km"] = distance_to_poi(
        out["dropoff_longitude"], out["dropoff_latitude"], jfk_lon, jfk_lat
    )

    out["pickup_to_lga_km"] = distance_to_poi(
        out["pickup_longitude"], out["pickup_latitude"], lga_lon, lga_lat
    )
    out["dropoff_to_lga_km"] = distance_to_poi(
        out["dropoff_longitude"], out["dropoff_latitude"], lga_lon, lga_lat
    )

    out["pickup_to_manhattan_km"] = distance_to_poi(
        out["pickup_longitude"], out["pickup_latitude"], mh_lon, mh_lat
    )
    out["dropoff_to_manhattan_km"] = distance_to_poi(
        out["dropoff_longitude"], out["dropoff_latitude"], mh_lon, mh_lat
    )

    # optional airport flags
    airport_radius_km = 2.0

    out["is_jfk_trip"] = (
        (out["pickup_to_jfk_km"] <= airport_radius_km) |
        (out["dropoff_to_jfk_km"] <= airport_radius_km)
    ).astype(int)

    out["is_lga_trip"] = (
        (out["pickup_to_lga_km"] <= airport_radius_km) |
        (out["dropoff_to_lga_km"] <= airport_radius_km)
    ).astype(int)

    out["pickup_in_manhattan_core"] = (
        (out["pickup_longitude"].between(-74.02, -73.93)) &
        (out["pickup_latitude"].between(40.70, 40.82))
    ).astype(int)

    out["dropoff_in_manhattan_core"] = (
        (out["dropoff_longitude"].between(-74.02, -73.93)) &
        (out["dropoff_latitude"].between(40.70, 40.82))
    ).astype(int)

    out["dist_x_jfk"] = out["manhattan_km"] * out["is_jfk_trip"]
    out["dist_x_lga"] = out["manhattan_km"] * out["is_lga_trip"]

    out["is_short_trip"] = (out["manhattan_km"] < 1).astype(int)
    out["is_long_trip"] = (out["manhattan_km"] >= 15).astype(int)
    out["dist_x_long_trip"] = out["manhattan_km"] * out["is_long_trip"]

    out["dist_x_rush"] = out["manhattan_km"] * out["is_rush_hour"]
    out["dist_x_night"] = out["manhattan_km"] * out["is_night"]

    out["hour_sin"] = np.sin(2 * np.pi * out["pickup_hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["pickup_hour"] / 24)

    return out

In [4]:
# Show the table: 
train_feat = add_features(train)

Clean data:

In [5]:
NYC_BOUNDS = {
    "lat_min": 40.0, "lat_max": 42.0,
    "lon_min": -75.0, "lon_max": -72.0
}

def cleaning_pipeline(df: pd.DataFrame, is_train: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      cleaned_df
      report_df: rule-wise removal report
    """
    x = df.copy()
    report = []

    def apply_rule(mask_keep: pd.Series, rule_name: str):
        nonlocal x, report
        before = len(x)
        x = x.loc[mask_keep].copy()
        after = len(x)
        report.append({
            "rule": rule_name,
            "removed": before - after,
            "removed_pct": (before - after) / before * 100.0 if before else 0.0,
            "remaining": after
        })

    # 0) drop rows with essential missing
    essential = ["pickup_datetime","pickup_longitude","pickup_latitude","dropoff_longitude","dropoff_latitude","passenger_count"]
    if is_train:
        essential += ["fare_amount"]

    before = len(x)
    x = x.dropna(subset=essential).copy()
    report.append({"rule":"dropna_essential", "removed": before-len(x), "removed_pct": (before-len(x))/before*100.0, "remaining": len(x)})

    # 1) passenger_count plausible
    apply_rule((x["passenger_count"] >= 1) & (x["passenger_count"] <= 6), "passenger_count_1_to_6")

    # 2) geo bounds for pickup/dropoff
    b = NYC_BOUNDS
    geo_keep = (
        x["pickup_latitude"].between(b["lat_min"], b["lat_max"]) &
        x["dropoff_latitude"].between(b["lat_min"], b["lat_max"]) &
        x["pickup_longitude"].between(b["lon_min"], b["lon_max"]) &
        x["dropoff_longitude"].between(b["lon_min"], b["lon_max"])
    )
    apply_rule(geo_keep, "geo_within_nyc_bounds")

    # 3) train-only fare sanity
    if is_train:
        apply_rule(x["fare_amount"] > 0, "fare_amount_positive")
        q_hi = x["fare_amount"].quantile(0.999)
        apply_rule(x["fare_amount"] <= q_hi, f"fare_amount_le_q999({q_hi:.2f})")

    # 4) add features then distance sanity
    x = add_features(x)

    # drop datetime parse failures
    apply_rule(x["pickup_datetime"].notna(), "pickup_datetime_parsed")

    # remove zero/near-zero distance but high fare noise 
    # added manhattan_km > 0 condition to avoid keeping zero-distance outliers with high fare
    apply_rule((x["haversine_km"] > 0) & (x["manhattan_km"] > 0), "distance_gt_0")

    # trim extreme distance
    d_hi = x["haversine_km"].quantile(0.999)
    apply_rule(x["haversine_km"] <= d_hi, f"distance_le_q999({d_hi:.2f}km)")

    report_df = pd.DataFrame(report)
    return x, report_df

train_clean, train_report = cleaning_pipeline(train, is_train=True)
test_clean, _ = cleaning_pipeline(test, is_train=False)

train_clean.shape, test_clean.shape, train_report

((9631370, 42),
 (9819, 41),
                          rule  removed  removed_pct  remaining
 0            dropna_essential       69     0.000690    9999931
 1      passenger_count_1_to_6    35280     0.352802    9964651
 2       geo_within_nyc_bounds   210081     2.108262    9754570
 3        fare_amount_positive      599     0.006141    9753971
 4  fare_amount_le_q999(78.75)     9701     0.099457    9744270
 5      pickup_datetime_parsed        0     0.000000    9744270
 6               distance_gt_0   103258     1.059679    9641012
 7   distance_le_q999(23.27km)     9642     0.100010    9631370)

In [6]:
# same as nyc bounds
LAT_MIN, LAT_MAX = 40.5, 41.0
LON_MIN, LON_MAX = -74.3, -73.7

def add_grid(df, lat_col, lon_col, n_lat=60, n_lon=60, prefix=""):
    out = df.copy()

    lat = out[lat_col].clip(LAT_MIN, LAT_MAX)
    lon = out[lon_col].clip(LON_MIN, LON_MAX)

    dlat = (LAT_MAX - LAT_MIN) / n_lat
    dlon = (LON_MAX - LON_MIN) / n_lon

    out[f"{prefix}grid_i"] = ((lat - LAT_MIN) / dlat).astype(int).clip(0, n_lat-1)
    out[f"{prefix}grid_j"] = ((lon - LON_MIN) / dlon).astype(int).clip(0, n_lon-1)

    out[f"{prefix}cell_id"] = out[f"{prefix}grid_i"] * n_lon + out[f"{prefix}grid_j"]
    return out

df = train_clean.copy()

df = add_grid(df, "pickup_latitude",  "pickup_longitude",  n_lat=20, n_lon=20, prefix="pu_")
df = add_grid(df, "dropoff_latitude", "dropoff_longitude", n_lat=20, n_lon=20, prefix="do_")

pu_counts = df["pu_cell_id"].value_counts()
print("pickup non-empty cells:", (pu_counts>0).sum())
print("pickup median per cell:", pu_counts.median())
print("pickup <20 samples ratio:", (pu_counts<20).mean())

# 先切分（确保统计量只用 train 计算）
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

def add_cell_stats(train_df, other_df, cell_col, target_col="fare_amount", prefix=""):
    stats = train_df.groupby(cell_col)[target_col].agg(["mean", "count"]).rename(
        columns={"mean": f"{prefix}mean", "count": f"{prefix}count"}
    )
    # merge
    other_df = other_df.join(stats, on=cell_col)
    return other_df, stats

# 给 train / val 都加 pickup cell stats
train_df, pu_stats = add_cell_stats(train_df, train_df, "pu_cell_id", prefix="pu_fare_")
val_df  = val_df.join(pu_stats, on="pu_cell_id")

# 给 train / val 都加 dropoff cell stats
train_df, do_stats = add_cell_stats(train_df, train_df, "do_cell_id", prefix="do_fare_")
val_df  = val_df.join(do_stats, on="do_cell_id")

# 缺失（val 中出现 train 没见过的 cell）补默认值
for c in ["pu_fare_mean","pu_fare_count","do_fare_mean","do_fare_count"]:
    train_df[c] = train_df[c].fillna(train_df["fare_amount"].mean() if "mean" in c else 0)
    val_df[c]   = val_df[c].fillna(train_df["fare_amount"].mean() if "mean" in c else 0)

pickup non-empty cells: 400
pickup median per cell: 27.0
pickup <20 samples ratio: 0.385


Features:

In [7]:
# features:
base_features = ["passenger_count", "pickup_hour", "pickup_dow", "is_weekend", "is_night", "haversine_km"]

grid_features = ["pu_grid_i","pu_grid_j","do_grid_i","do_grid_j"]

manhattan_features = ["manhattan_km", "distance_ratio_manhattan_haversine"]

bearing_features = ["trip_bearing", "bearing_sin", "bearing_cos"]

poi_features = ["pickup_to_jfk_km", "dropoff_to_jfk_km",
    "pickup_to_lga_km", "dropoff_to_lga_km", "pickup_to_manhattan_km",
    "dropoff_to_manhattan_km", "is_jfk_trip","is_lga_trip"]

interaction_features = ["dist_x_jfk", "dist_x_lga", "dist_x_rush", "dist_x_night"]

trip_length_features = ["is_long_trip", "dist_x_long_trip"]

year_features = ["pickup_month", "pickup_year", "year_index", "year_month_index"]

core_features = ["pickup_in_manhattan_core", "dropoff_in_manhattan_core"]

FULL = base_features + grid_features + manhattan_features + poi_features + interaction_features + trip_length_features + bearing_features + year_features + core_features

Define XGBoost function:

In [15]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def train_eval(features, name):
    X_tr = train_df[features]
    y_tr = train_df["fare_amount"]
    X_va = val_df[features]
    y_va = val_df["fare_amount"]

    model = XGBRegressor(
        n_estimators=4000,
        learning_rate=0.03,
        max_depth=8,
        min_child_weight=8,
        subsample=1.0,
        colsample_bytree=0.8,
        reg_lambda=5.0,
        reg_alpha=0.1,
        random_state=42,
        n_jobs=-1,
        objective="reg:squarederror",
        tree_method="hist",
        device = "cuda"
    )
    model.fit(X_tr, y_tr)
    pred = model.predict(X_va)
    print(f"{name} RMSE: {rmse(y_va.to_numpy(), pred):.4f}")
    print(f"{name} MAE : {mean_absolute_error(y_va.to_numpy(), pred):.4f}")
    return model

In [16]:
print("\n=== FULL Test ===")
_ = train_eval(FULL, "XCB FULL Features")


=== FULL Test ===


MemoryError: Unable to allocate 588. MiB for an array with shape (10, 7705096) and data type int64

Use device = "cuda" to get the result.

In [11]:
def tune_xgb_best_params(
    train_df,
    val_df,
    features,
    target="fare_amount",
    param_grid=None,
    fixed_params=None,
    sort_by="rmse",
    refit_on_all=False,
    verbose=True
):
    """
    Exhaustive search for the best XGBoost parameter combination on the current train/val split.

    Parameters
    ----------
    train_df : pd.DataFrame
    val_df : pd.DataFrame
    features : list[str]
        Feature columns to use.
    target : str, default="fare_amount"
    param_grid : dict, default=None
        Example:
        {
            "n_estimators": [400, 800, 1200],
            "learning_rate": [0.03, 0.05, 0.1],
            "max_depth": [8, 12, 16],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0],
            "min_child_weight": [1, 3, 5],
            "reg_alpha": [0.0, 0.1, 0.3],
            "reg_lambda": [1.0, 3.0, 5.0]
        }
    fixed_params : dict, default=None
        Parameters always passed into XGBRegressor.
    sort_by : str, default="rmse"
        "rmse" or "mae"
    refit_on_all : bool, default=False
        If True, refit best model on train_df + val_df.
    verbose : bool, default=True

    Returns
    -------
    best_model : fitted XGBRegressor
    results_df : pd.DataFrame
    best_params : dict
    """
    if param_grid is None:
        param_grid = {
            "n_estimators": [4000],
            "learning_rate": [0.03, 0.05],
            "max_depth": [8, 14],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0],
            "min_child_weight": [5, 7],
            "reg_alpha": [0.1, 0.3],
            "reg_lambda": [1.0, 3.0, 5.0],
        }

    if fixed_params is None:
        fixed_params = {
            "objective": "reg:squarederror",
            "tree_method": "hist",
            "random_state": 42,
            "n_jobs": -1,
            "device":"cuda"
        }

    if sort_by not in ["rmse", "mae"]:
        raise ValueError("sort_by must be 'rmse' or 'mae'")

    X_tr = train_df[features]
    y_tr = train_df[target]
    X_va = val_df[features]
    y_va = val_df[target]

    grid_keys = list(param_grid.keys())
    grid_values = [param_grid[k] for k in grid_keys]

    results = []
    best_score = np.inf
    best_params = None
    best_model = None

    total = np.prod([len(v) for v in grid_values])
    count = 0

    for values in product(*grid_values):
        count += 1
        trial_params = dict(zip(grid_keys, values))
        model_params = {**fixed_params, **trial_params}

        model = XGBRegressor(**model_params)
        model.fit(X_tr, y_tr)

        pred = model.predict(X_va)
        cur_rmse = rmse(y_va.to_numpy(), pred)
        cur_mae = mean_absolute_error(y_va.to_numpy(), pred)

        row = {**trial_params, "rmse": cur_rmse, "mae": cur_mae}
        results.append(row)

        score = cur_rmse if sort_by == "rmse" else cur_mae
        if score < best_score:
            best_score = score
            best_params = trial_params.copy()
            best_model = model

        if verbose:
            print(f"[{count}/{total}] rmse={cur_rmse:.6f}, mae={cur_mae:.6f}, params={trial_params}")

    results_df = pd.DataFrame(results).sort_values(sort_by).reset_index(drop=True)

    if refit_on_all:
        best_model = XGBRegressor(**{**fixed_params, **best_params})
        full_df = pd.concat([train_df, val_df], axis=0)
        best_model.fit(full_df[features], full_df[target])

    if verbose:
        print("\nBest params:")
        print(best_params)
        print(f"Best {sort_by.upper()}: {best_score:.6f}")

    return best_model, results_df, best_params

In [12]:
best_model, tuning_results, best_params = tune_xgb_best_params(
    train_df=train_df,
    val_df=val_df,
    features=FULL,
    sort_by="rmse",
    refit_on_all=False,
    verbose=True
)

tuning_results.head(10)

[1/192] rmse=2.915382, mae=1.398148, params={'n_estimators': 4000, 'learning_rate': 0.03, 'max_depth': 8, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 5, 'reg_alpha': 0.1, 'reg_lambda': 1.0}
[2/192] rmse=2.915456, mae=1.398982, params={'n_estimators': 4000, 'learning_rate': 0.03, 'max_depth': 8, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 5, 'reg_alpha': 0.1, 'reg_lambda': 3.0}
[3/192] rmse=2.913567, mae=1.398940, params={'n_estimators': 4000, 'learning_rate': 0.03, 'max_depth': 8, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 5, 'reg_alpha': 0.1, 'reg_lambda': 5.0}
[4/192] rmse=2.914891, mae=1.398652, params={'n_estimators': 4000, 'learning_rate': 0.03, 'max_depth': 8, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 5, 'reg_alpha': 0.3, 'reg_lambda': 1.0}
[5/192] rmse=2.914842, mae=1.398509, params={'n_estimators': 4000, 'learning_rate': 0.03, 'max_depth': 8, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weig

XGBoostError: [20:38:44] C:\actions-runner\_work\xgboost\xgboost\src\common\device_vector.cu:23: Memory allocation error on worker 0: bad allocation: cudaErrorMemoryAllocation: out of memory
- Free memory: 5.0752GB
- Requested memory: 1GB


XCB FULL Features RMSE: 2.9252
XCB FULL Features MAE : 1.3877

In [14]:
best_model

NameError: name 'best_model' is not defined

Catboost Test:

In [ ]:
def train_evl_catboost(X, y):
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=16,
        loss_function="RMSE",
        verbose=0,
        devices="cuda"
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, pred))
    mae = mean_absolute_error(y_val, pred)

    print(f"CatBoost RMSE: {rmse:.4f}")
    print(f"CatBoost MAE : {mae:.4f}")

    return model

In [ ]:
X = train_df[FULL].copy()
y = train_df["fare_amount"].copy()

X = X.drop(columns=["key", "pickup_datetime"], errors="ignore")
X = X.fillna(0)

train_evl_catboost(X, y)

CatBoost RMSE: 2.9567
CatBoost MAE : 1.4206


CatBoostRegressor(depth=16, iterations=1000, learning_rate=0.05, loss_function='RMSE', verbose=0)

CatBoost RMSE: 2.9567
CatBoost MAE : 1.4206